Используйте базу акций Лукойла.

Сделайте несколько усовершенствований в предсказании временного ряда.
Добавьте к исходному сигналу расширенные данные:

* попарные разности каналов
* модули попарных разностей каналов
* попарные произведения каналов
* обратное значение каналов x_new = 1/(x + 1e-3)
* первые производные каналов (x[n] - x[n-1])
* вторые производные каналов (x[n] - 2*x[n-1] + x[n-2])

Примените абсолютно новый подход. Сделайте бОльший “просмотр сети в прошлое”, при формировании входного сигнала используйте:

* 100 точек с шагом назад по 1,

* 100 точек с шагом назад по 10 (или сами точки, или среднее по отрезку в 10 точек).
* Объедините эти точки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

TICKER = 'LKOH.ME'
START_DATE = '2010-01-01'
SAVE_FILE = '1luk.csv'
TARGET_COL = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

quotes = yf.download(TICKER, start=START_DATE)

if isinstance(quotes.columns, pd.MultiIndex):
    close_series = quotes[('Close', TICKER)]
else:
    close_series = quotes['Close']

close_frame = close_series.to_frame(name='CLOSE')
close_frame.to_csv(SAVE_FILE)

print(f'Данные скачаны и сохранены в {SAVE_FILE}')


def expand_market_features(values):
    values = np.asarray(values, dtype=np.float32)
    feature_blocks = [values[:, i] for i in range(values.shape[1])]

    for i in range(values.shape[1]):
        for j in range(i + 1, values.shape[1]):
            first = values[:, i]
            second = values[:, j]
            feature_blocks.extend([first - second, np.abs(first - second), first * second])

    for i in range(values.shape[1]):
        feature_blocks.append(1 / (values[:, i] + 1e-3))

    base_matrix = np.vstack(feature_blocks).T
    first_diff = np.diff(base_matrix, axis=0, prepend=base_matrix[:1])
    second_diff = np.diff(first_diff, axis=0, prepend=first_diff[:1])

    return np.hstack([base_matrix, first_diff, second_diff]).astype(np.float32)


raw_values = quotes.values.astype(np.float32)
prepared_values = expand_market_features(raw_values)

print(f'Количество признаков после расширения: {prepared_values.shape[1]}')

scaler = StandardScaler()
data_scaled = scaler.fit_transform(prepared_values).astype(np.float32)

In [ ]:
def make_windows(matrix, target_col, window=100, step=1):
    x_data = []
    y_data = []

    for index in range(window, len(matrix)):
        x_data.append(matrix[index - window:index:step])
        y_data.append(matrix[index, target_col])

    return np.asarray(x_data, dtype=np.float32), np.asarray(y_data, dtype=np.float32)


def time_split(x_data, y_data, train_ratio=0.8):
    split_index = int(len(x_data) * train_ratio)
    return x_data[:split_index], x_data[split_index:], y_data[:split_index], y_data[split_index:]


X_pro, y_pro = make_windows(
    data_scaled,
    target_col=TARGET_COL,
    window=100,
    step=1
)

x_train, x_test, y_train, y_test = time_split(X_pro, y_pro, train_ratio=0.8)

train_ds = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
)

test_ds = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

x_train.shape, x_test.shape

In [ ]:
class ProSeriesNet(nn.Module):
    def __init__(self, features_count):
        super().__init__()
        self.conv = nn.Conv1d(features_count, 128, kernel_size=5)
        self.norm = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(128, 64, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.norm(self.conv(x)))
        x = x.permute(0, 2, 1)
        _, (hidden, _) = self.lstm(x)
        x = self.drop(hidden[-1])
        return self.head(x)


model_pro = ProSeriesNet(x_train.shape[2]).to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_pro.parameters(), lr=1e-4)


def run_epoch(model, loader, loss_function, optim=None):
    training_mode = optim is not None
    model.train(training_mode)

    total_loss = 0
    total_mae = 0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if training_mode:
            optim.zero_grad()

        output = model(xb)
        loss = loss_function(output, yb)

        if training_mode:
            loss.backward()
            optim.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_mae += torch.mean(torch.abs(output.detach() - yb)).item() * batch_size
        total_count += batch_size

    return total_loss / total_count, total_mae / total_count


history = {
    'loss': [],
    'mae': [],
    'val_loss': [],
    'val_mae': []
}

for epoch in range(1, 31):
    train_loss, train_mae = run_epoch(model_pro, train_loader, loss_fn, optimizer)

    with torch.no_grad():
        val_loss, val_mae = run_epoch(model_pro, test_loader, loss_fn)

    history['loss'].append(train_loss)
    history['mae'].append(train_mae)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)

    print(
        f'Эпоха {epoch:02d}/30 | '
        f'loss: {train_loss:.6f} | mae: {train_mae:.6f} | '
        f'val_loss: {val_loss:.6f} | val_mae: {val_mae:.6f}'
    )

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(history['mae'], label='MAE на обучении')
plt.plot(history['val_mae'], label='MAE на проверке')
plt.title('Ошибка Pro-модели на PyTorch')
plt.xlabel('Эпоха')
plt.ylabel('MAE')
plt.legend()
plt.grid()
plt.show()

In [ ]:
def restore_target(values, scaler_object, column_id=TARGET_COL):
    values = np.asarray(values).reshape(-1)
    blank = np.zeros((len(values), scaler_object.n_features_in_), dtype=np.float32)
    blank[:, column_id] = values
    restored = scaler_object.inverse_transform(blank)
    return restored[:, column_id]


def predict_batches(model, loader):
    model.eval()
    result = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            result.append(model(xb).cpu().numpy())

    return np.vstack(result).reshape(-1)


pred_scaled = predict_batches(model_pro, test_loader)

y_pred_orig = restore_target(pred_scaled, scaler, TARGET_COL)
y_true_orig = restore_target(y_test, scaler, TARGET_COL)

mae_price = np.mean(np.abs(y_true_orig - y_pred_orig))
mean_price = np.mean(np.abs(y_true_orig))
error_percent = mae_price / mean_price * 100

print(f'MAE в исходной шкале: {mae_price:.2f}')
print(f'Средняя ошибка: {error_percent:.2f}%')

plt.figure(figsize=(15, 7))
plt.plot(y_true_orig, label='Реальная цена (Close)')
plt.plot(y_pred_orig, label='Предсказание Pro-модели PyTorch', alpha=0.8)
plt.title('Результат Pro-модели с расширенными признаками')
plt.legend()
plt.grid()
plt.show()